# 12 · Structured Streaming — Janelas e Watermarks

🎯 **Objetivo:** aplicar `tumbling`, `sliding` e `session windows`, e `watermarks`, sobre um fluxo de vendas simulado — com a mesma API de DataFrame dos notebooks anteriores, agora sobre dados que nunca terminam de chegar.

**Teoria:** docs/09-spark-streaming.md

Rodamos em `local[*]`, sem Docker e sem Kafka — o mesmo modo dos notebooks 01-05. Este laboratório não tem um serviço Kafka disponível, então simulamos o fluxo com o **file source**: uma pasta que recebe novos arquivos aos poucos, exatamente como um tópico receberia novas mensagens. O código de janelas e watermarks deste notebook é **idêntico** ao que você escreveria com `format("kafka")` — só a fonte muda, exatamente como o módulo teórico enfatiza.

📌 Nos notebooks 01-11, toda pergunta tinha uma resposta **final** — o Parquet da Bronze tem começo e fim. Aqui não: `vendas_stream.count()` nem existe, porque "quantas vendas no total?" não tem resposta em um fluxo infinito. A pergunta certa vira "quantas vendas **em qual janela de tempo**?" — é isso que este notebook resolve.

---
### 🔤 O que você vai praticar

1. **File source** — simular um stream sem Kafka, escrevendo arquivos incrementalmente numa pasta "landing"
2. **Tumbling windows** — `window(coluna, "5 minutes")`: janelas fixas, contíguas, sem sobreposição
3. **Sliding windows** — `window(coluna, "10 minutes", "5 minutes")`: janelas fixas e sobrepostas
4. **Session windows** — `session_window(coluna, "30 minutes")`: janelas dinâmicas por atividade de uma chave
5. **Watermarks** — `withWatermark(coluna, "10 minutes")`: até quando vale a pena esperar por dados atrasados
6. **Modos de saída** — `complete`, `update` e `append`, e por que cada tipo de janela combina melhor com um (ou mais) desses modos

Vamos simular o fluxo!

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("app-01")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.executor.memory", "2g")
    .config("spark.executor.cores", "2")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")  # silencia o ruído de INFO/WARN de cada micro-batch

spark

## Simulando um stream sem Kafka: o file source

O **file source** trata uma pasta como se fosse um tópico: cada novo arquivo que aparece nela é lido como um novo lote de linhas anexadas à *Input Table* infinita do Structured Streaming. É a forma oficialmente suportada (e mais usada em tutoriais e testes) de aprender e validar lógica de streaming sem precisar de um broker de verdade rodando ao lado.

Vamos construir um "produtor" bem simples: uma função Python que escreve um lote de eventos de venda como um arquivo JSON Lines numa pasta `landing`. O Spark, do outro lado, fica de olho nessa pasta com `readStream`.

⚠️ **Atenção:** assim como no `docs`/módulo teórico, streaming **exige schema explícito** — sem `inferSchema`, porque o Spark não pode escanear um arquivo que ainda nem chegou por completo.

💡 **Truque de robustez:** escrevemos cada arquivo primeiro com um sufixo `.tmp` e só então damos `rename()` para o nome final. O rename é atômico no sistema de arquivos — evita que o file source tente ler um arquivo pela metade, no meio de uma escrita.

In [ ]:
import json
import os
import shutil
import time
import uuid
from datetime import datetime, timedelta
from pyspark.sql.types import DoubleType, StringType, StructField, StructType, TimestampType

# Toda a simulação vive sob esta pasta — limpa a cada execução do notebook,
# para que o resultado seja sempre reproduzível.
BASE = "../data/streaming/nb12"
shutil.rmtree(BASE, ignore_errors=True)


def nova_pasta_landing(nome: str) -> str:
    caminho = f"{BASE}/{nome}"
    os.makedirs(caminho, exist_ok=True)
    return caminho


def emitir_lote(eventos: list[dict], pasta: str) -> None:
    """Publica um lote de eventos como um novo arquivo JSON Lines na pasta 'landing'."""
    nome_arquivo = f"{uuid.uuid4().hex}.json"
    caminho_tmp = f"{pasta}/.{nome_arquivo}.tmp"
    caminho_final = f"{pasta}/{nome_arquivo}"
    with open(caminho_tmp, "w") as f:
        for evento in eventos:
            f.write(json.dumps(evento) + "\n")
    os.rename(caminho_tmp, caminho_final)  # rename atômico: só agora o arquivo "existe" para o Spark
    print(f"📨 lote de {len(eventos)} evento(s) publicado em {pasta.split('/')[-1]}/{nome_arquivo}")


# Mesmo domínio de negócio dos notebooks 02-04 (vendas), agora como evento de streaming
schema_vendas_stream = StructType([
    StructField("id_venda", StringType()),
    StructField("id_funcionario", StringType()),
    StructField("valor", DoubleType()),
    StructField("timestamp_venda", TimestampType()),  # nosso event time
])

# Data-base fixa e sintética: controlamos o event_time de cada evento por completo,
# então o resultado é 100% determinístico — independe de QUANDO você rodar este notebook.
BASE_TIME = datetime(2026, 1, 1, 14, 0, 0)

## Tumbling Windows

Janelas fixas, contíguas e **sem sobreposição** — cada evento cai em exatamente uma janela. É o tipo mais simples e mais usado em relatórios de negócio: "vendas por hora", "erros por minuto".

#### 💡 **Exemplo 1:** Total vendido em janelas de 5 minutos

Vamos abrir um stream sobre a pasta `landing_tumbling` e agregar com `groupBy(window(...))`. Como o número de janelas em aberto é pequeno, usamos `outputMode("complete")` — a cada micro-batch, o Spark reescreve a tabela de resultado inteira no sink `memory`, o que nos deixa consultar o estado atual a qualquer momento com `spark.sql(...)`.

📌 Repare que o `event_time` de cada venda é definido **por nós**, na hora de montar o evento — não é o relógio da máquina. Isso significa que as janelas `[14:00,14:05)` e `[14:05,14:10)` abaixo são calculadas sobre esse tempo simulado, não sobre o tempo real de execução do notebook.

In [ ]:
from pyspark.sql.functions import col, count, window, sum as spark_sum

pasta_tumbling = nova_pasta_landing("landing_tumbling")

# readStream em vez de read — a partir daqui, é DataFrame comum: filter, groupBy, agg...
vendas_stream = (
    spark.readStream
    .format("json")
    .schema(schema_vendas_stream)
    .load(pasta_tumbling)
)
print(f"vendas_stream.isStreaming = {vendas_stream.isStreaming}")  # True — diferente de todo DataFrame dos notebooks 01-11

total_por_janela = (
    vendas_stream
    .groupBy(window(col("timestamp_venda"), "5 minutes"))
    .agg(
        spark_sum("valor").alias("total_vendido"),
        count("*").alias("qtd_vendas"),
    )
)

query_tumbling = (
    total_por_janela.writeStream
    .format("memory")          # sink "memory": guarda o resultado numa tabela SQL temporária, ótimo para explorar em notebook
    .queryName("tumbling")      # nome da tabela temporária: spark.sql("select * from tumbling")
    .outputMode("complete")
    .trigger(processingTime="2 seconds")
    .start()
)

In [ ]:
# P1, P2 e P3 caem todos dentro de [14:00, 14:05) — mesmo exemplo traçado na teoria
emitir_lote([
    {"id_venda": "P1", "id_funcionario": "F1", "valor": 120.0,
     "timestamp_venda": (BASE_TIME + timedelta(minutes=1, seconds=15)).isoformat()},
    {"id_venda": "P2", "id_funcionario": "F1", "valor": 80.0,
     "timestamp_venda": (BASE_TIME + timedelta(minutes=3, seconds=40)).isoformat()},
    {"id_venda": "P3", "id_funcionario": "F1", "valor": 200.0,
     "timestamp_venda": (BASE_TIME + timedelta(minutes=4, seconds=59)).isoformat()},
], pasta_tumbling)

time.sleep(5)  # dá tempo para o próximo trigger de 2s processar o arquivo recém-chegado
spark.sql("SELECT window, total_vendido, qtd_vendas FROM tumbling ORDER BY window").show(truncate=False)

In [ ]:
# P4 abre a janela seguinte [14:05, 14:10); P5 continua nela
emitir_lote([
    {"id_venda": "P4", "id_funcionario": "F1", "valor": 150.0,
     "timestamp_venda": (BASE_TIME + timedelta(minutes=5, seconds=2)).isoformat()},
    {"id_venda": "P5", "id_funcionario": "F1", "valor": 90.0,
     "timestamp_venda": (BASE_TIME + timedelta(minutes=8, seconds=20)).isoformat()},
], pasta_tumbling)

time.sleep(5)
spark.sql("SELECT window, total_vendido, qtd_vendas FROM tumbling ORDER BY window").show(truncate=False)

📌 **Resultado:** `[14:00,14:05) → R$ 400,00` (3 vendas) e `[14:05,14:10) → R$ 240,00` (2 vendas) — os **mesmos** números do exemplo traçado na teoria, agora produzidos por uma query de streaming rodando de verdade, não numa tabela manual.

- P3 (14:04:59) e P4 (14:05:02) estão a **3 segundos** de distância um do outro, mas caem em janelas diferentes — o limite `14:05:00` é exclusivo à direita/inclusivo à esquerda, exatamente como a definição formal prevê.
- Com `outputMode("complete")`, cada consulta ao sink `memory` mostra o estado **inteiro** até aquele momento: a segunda consulta já traz as duas janelas, não só a nova.

⚠️ **Sobre os `time.sleep()` deste notebook:** eles pausam nosso código Python só para dar tempo do **próximo trigger de 2 segundos** processar o arquivo recém-chegado antes de consultarmos o resultado — não têm nenhuma relação com a duração das janelas, que é sempre calculada sobre o `timestamp_venda` (event time) de cada evento.

In [ ]:
query_tumbling.stop()

#### 💡 **Exemplo 2:** Painel ao vivo por vendedor — `groupBy` com janela **e** chave

`window(...)` é só mais uma coluna dentro do `groupBy` — nada impede de combiná-la com uma coluna de negócio, exatamente como fizemos com `groupBy("setor", "ano", "mes")` no notebook 03. Vamos simular um painel que qualquer gerente comercial ia querer ver ao vivo: total vendido **por vendedor, a cada 5 minutos**.

In [ ]:
pasta_painel = nova_pasta_landing("landing_painel_vendedor")

vendas_painel_stream = (
    spark.readStream.format("json").schema(schema_vendas_stream).load(pasta_painel)
)

painel_por_vendedor = (
    vendas_painel_stream
    .groupBy(window(col("timestamp_venda"), "5 minutes"), col("id_funcionario"))
    .agg(spark_sum("valor").alias("total_vendido"), count("*").alias("qtd_vendas"))
)

query_painel = (
    painel_por_vendedor.writeStream
    .format("memory")
    .queryName("painel_vendedor")
    .outputMode("complete")
    .trigger(processingTime="2 seconds")
    .start()
)

# Dois vendedores, mesma janela [14:00,14:05)
emitir_lote([
    {"id_venda": "V1", "id_funcionario": "F1", "valor": 100.0,
     "timestamp_venda": (BASE_TIME + timedelta(minutes=1)).isoformat()},
    {"id_venda": "V2", "id_funcionario": "F1", "valor": 50.0,
     "timestamp_venda": (BASE_TIME + timedelta(minutes=2)).isoformat()},
    {"id_venda": "V3", "id_funcionario": "F2", "valor": 300.0,
     "timestamp_venda": (BASE_TIME + timedelta(minutes=1, seconds=30)).isoformat()},
], pasta_painel)

time.sleep(5)
spark.sql(
    "SELECT window, id_funcionario, total_vendido, qtd_vendas "
    "FROM painel_vendedor ORDER BY window, id_funcionario"
).show(truncate=False)

In [ ]:
# Janela seguinte [14:05,14:10) — F1 vende pouco, F2 dispara
emitir_lote([
    {"id_venda": "V4", "id_funcionario": "F1", "valor": 70.0,
     "timestamp_venda": (BASE_TIME + timedelta(minutes=6)).isoformat()},
    {"id_venda": "V5", "id_funcionario": "F2", "valor": 500.0,
     "timestamp_venda": (BASE_TIME + timedelta(minutes=7)).isoformat()},
    {"id_venda": "V6", "id_funcionario": "F2", "valor": 20.0,
     "timestamp_venda": (BASE_TIME + timedelta(minutes=8)).isoformat()},
], pasta_painel)

time.sleep(5)
spark.sql(
    "SELECT window, id_funcionario, total_vendido, qtd_vendas "
    "FROM painel_vendedor ORDER BY window, id_funcionario"
).show(truncate=False)

query_painel.stop()

📌 **Entendendo a saída:** cada linha agora é uma combinação **janela × vendedor** — F1 vendeu R\$150 em `[14:00,14:05)` e apenas R\$70 na janela seguinte, enquanto F2 disparou de R\$300 para R\$520. Um painel assim atualiza a cada 5 minutos sem que ninguém precise agendar um job de relatório — a query fica rodando, incorporando cada novo arquivo (na vida real, cada nova mensagem do tópico Kafka) automaticamente.

🧠 **Por quê `outputMode("complete")` continua funcionando aqui?** O número de combinações janela×vendedor ainda é pequeno (poucos vendedores, poucas janelas abertas por vez) — o custo de reescrever a tabela inteira a cada micro-batch continua desprezível. Isso deixaria de ser viável com milhares de vendedores e janelas simultâneas; nesse caso, `update` seria a escolha (só as linhas que mudaram).

## Sliding Windows

Mesma duração fixa do tumbling, mas o passo de avanço (*slide*) é **menor** que a duração — janelas consecutivas se sobrepõem, e um evento pode contribuir para **múltiplas** janelas ao mesmo tempo. A diferença de sintaxe é um único argumento extra em `window(...)`.

#### 💡 **Exemplo 3:** Média móvel — os mesmos 5 eventos, agora sobrepostos

Vamos reaproveitar **exatamente os mesmos** P1-P5 do Exemplo 1, mas com uma janela de **10 minutos deslizando a cada 5 minutos**. A ideia é comparar diretamente com o resultado tumbling que já vimos.

In [ ]:
from pyspark.sql.functions import avg

pasta_sliding = nova_pasta_landing("landing_sliding")

vendas_sliding_stream = (
    spark.readStream.format("json").schema(schema_vendas_stream).load(pasta_sliding)
)

media_movel = (
    vendas_sliding_stream
    # window(coluna, duração, deslizamento) — único argumento novo em relação ao tumbling
    .groupBy(window(col("timestamp_venda"), "10 minutes", "5 minutes"))
    .agg(
        spark_sum("valor").alias("total_vendido"),
        avg("valor").alias("ticket_medio"),
        count("*").alias("qtd_vendas"),
    )
)

query_sliding = (
    media_movel.writeStream
    .format("memory")
    .queryName("sliding")
    .outputMode("complete")
    .trigger(processingTime="2 seconds")
    .start()
)

# Os MESMOS 5 eventos do Exemplo 1 — só a janela de agregação muda
emitir_lote([
    {"id_venda": "P1", "id_funcionario": "F1", "valor": 120.0,
     "timestamp_venda": (BASE_TIME + timedelta(minutes=1, seconds=15)).isoformat()},
    {"id_venda": "P2", "id_funcionario": "F1", "valor": 80.0,
     "timestamp_venda": (BASE_TIME + timedelta(minutes=3, seconds=40)).isoformat()},
    {"id_venda": "P3", "id_funcionario": "F1", "valor": 200.0,
     "timestamp_venda": (BASE_TIME + timedelta(minutes=4, seconds=59)).isoformat()},
    {"id_venda": "P4", "id_funcionario": "F1", "valor": 150.0,
     "timestamp_venda": (BASE_TIME + timedelta(minutes=5, seconds=2)).isoformat()},
    {"id_venda": "P5", "id_funcionario": "F1", "valor": 90.0,
     "timestamp_venda": (BASE_TIME + timedelta(minutes=8, seconds=20)).isoformat()},
], pasta_sliding)

time.sleep(5)
spark.sql(
    "SELECT window, total_vendido, ticket_medio, qtd_vendas FROM sliding ORDER BY window"
).show(truncate=False)

query_sliding.stop()

📌 **Comparando com o tumbling do Exemplo 1:**

| Janela | Total | Qtd |
|---|---|---|
| `[13:55,14:05)` | R\$ 400,00 | 3 |
| `[14:00,14:10)` | R\$ 640,00 | 5 |
| `[14:05,14:15)` | R\$ 240,00 | 2 |

Repare que a janela do meio, `[14:00,14:10)`, soma **R\$ 640,00 = R\$ 400,00 + R\$ 240,00** — exatamente as duas janelas tumbling que calculamos antes, somadas. Isso não é coincidência: com duração 10min e passo 5min, cada evento pertence a `10/5 = 2` janelas — a sliding window "cobre a transição" entre dois blocos tumbling adjacentes, suavizando o corte abrupto que o tumbling impõe em `14:05:00`.

🧠 **Por quê isso importa para negócio?** Se seu KPI é "detectar quedas bruscas de venda", uma janela tumbling pode "esconder" uma queda que aconteceu bem no meio dela. A sliding window, recalculada a cada 5 minutos sobre os últimos 10, reage mais rápido — ao custo de processar cada evento mais de uma vez (maior custo computacional, como já vimos na teoria).

## Session Windows

Diferente de tumbling e sliding — que são **alinhadas ao relógio**, com limites conhecidos de antemão — uma *session window* tem duração **variável**, definida pela atividade real de uma chave. A janela fica aberta enquanto novos eventos chegam dentro de um **gap** de inatividade; se o gap é excedido, a sessão fecha e a próxima começa do zero.

#### 💡 **Exemplo 4:** Sessão de vendas de um vendedor em plantão

Em vez da sessão de navegação clássica (cliques de um usuário), vamos sessionizar a **atividade de vendas** de um vendedor: enquanto ele fecha negócios a cada poucos minutos, é a mesma "sessão de vendas". Se ele fica mais de 30 minutos sem vender, consideramos que o plantão "esfriou" — a próxima venda abre uma sessão nova.

⚠️ **`session_window` exige `withWatermark()`** — sem isso, o Spark nunca saberia quando considerar uma sessão definitivamente encerrada. Vamos manter **duas** queries lendo a mesma pasta, para comparar dois jeitos de observar a mesma sessão:

- `sessoes_visao_atual` (`outputMode("complete")`): mostra o estado corrente, incluindo sessões **ainda abertas** — útil para um painel ao vivo.
- `sessoes_fechadas` (`outputMode("append")`): só mostra uma sessão quando ela está **definitivamente fechada** pelo watermark — útil para gravar em Parquet sem duplicar nem reescrever.

In [ ]:
from pyspark.sql.functions import max as spark_max, min as spark_min, session_window

pasta_sessao = nova_pasta_landing("landing_sessao")

vendas_sessao_stream = (
    spark.readStream.format("json").schema(schema_vendas_stream).load(pasta_sessao)
)

sessoes_de_venda = (
    vendas_sessao_stream
    .withWatermark("timestamp_venda", "15 minutes")   # obrigatório para session_window
    .groupBy(
        col("id_funcionario"),
        session_window(col("timestamp_venda"), "30 minutes"),   # gap de inatividade
    )
    .agg(
        count("*").alias("qtd_vendas"),
        spark_min("timestamp_venda").alias("inicio_sessao"),
        spark_max("timestamp_venda").alias("fim_ultima_venda"),
    )
)

query_sessao_atual = (
    sessoes_de_venda.writeStream
    .format("memory").queryName("sessoes_visao_atual")
    .outputMode("complete")
    .trigger(processingTime="2 seconds").start()
)
query_sessao_fechada = (
    sessoes_de_venda.writeStream
    .format("memory").queryName("sessoes_fechadas")
    .outputMode("append")     # só emite quando o watermark confirma que a sessão encerrou
    .trigger(processingTime="2 seconds").start()
)


def mostrar_sessoes(rotulo: str) -> None:
    print(f"=== {rotulo} ===")
    print("👁️  visão atual (complete, inclui sessões abertas):")
    spark.sql(
        "SELECT id_funcionario, session_window, qtd_vendas, inicio_sessao, fim_ultima_venda "
        "FROM sessoes_visao_atual ORDER BY session_window"
    ).show(truncate=False)
    print("🔒 sessões fechadas (append, só o que o watermark já confirmou):")
    spark.sql(
        "SELECT id_funcionario, session_window, qtd_vendas, inicio_sessao, fim_ultima_venda "
        "FROM sessoes_fechadas ORDER BY session_window"
    ).show(truncate=False)

In [ ]:
INICIO_PLANTAO = datetime(2026, 1, 1, 9, 0, 0)

# V1, V2, V3: vendas de F007 com gaps de 12min e 13min — bem dentro do limite de 30min
emitir_lote([
    {"id_venda": "V1", "id_funcionario": "F007", "valor": 500.0,
     "timestamp_venda": INICIO_PLANTAO.isoformat()},
    {"id_venda": "V2", "id_funcionario": "F007", "valor": 300.0,
     "timestamp_venda": (INICIO_PLANTAO + timedelta(minutes=12)).isoformat()},
    {"id_venda": "V3", "id_funcionario": "F007", "valor": 700.0,
     "timestamp_venda": (INICIO_PLANTAO + timedelta(minutes=25)).isoformat()},
], pasta_sessao)

time.sleep(5)
mostrar_sessoes("Depois de V1, V2, V3 — sessão A em andamento")

📌 A **visão atual** já mostra a sessão com 3 vendas, começando às `09:00:00`, com fim provisório recalculado a cada nova venda (`fim_ultima_venda` = 09:25:00 por enquanto — o fim **real** da sessão, `session_window.end`, é sempre `fim_ultima_venda + 30min`, e continuará mudando enquanto a sessão ficar aberta). Já a tabela de **sessões fechadas** está vazia — nada foi confirmado como encerrado ainda, porque o watermark mal avançou.

In [ ]:
# V4 chega 45 minutos depois de V3 — ultrapassa o gap de 30min: nova sessão!
emitir_lote([
    {"id_venda": "V4", "id_funcionario": "F007", "valor": 400.0,
     "timestamp_venda": (INICIO_PLANTAO + timedelta(minutes=70)).isoformat()},
], pasta_sessao)

time.sleep(5)
mostrar_sessoes("Depois de V4 (gap de 45min > 30min) — sessão A fecha, sessão B abre")

📌 **O momento-chave:** V4 (`10:10:00`) é o evento mais recente já visto, então o watermark avança para `10:10:00 - 15min = 09:55:00`. A sessão A termina exatamente em `09:55:00` (`fim_ultima_venda` 09:25:00 + 30min de gap) — como `09:55:00 ≤` watermark, a sessão A está **definitivamente fechada**, e é exatamente isso que aparece agora na tabela `sessoes_fechadas`: **uma única linha**, emitida **uma única vez**, com as 3 vendas.

Ao mesmo tempo, a visão atual já mostra a sessão B nascendo, com 1 venda e fim provisório em `10:40:00` — é o mesmo evento (V4) que fecha uma sessão **e** abre a próxima.

In [ ]:
# V5 continua a sessão B (gap de só 10min)
emitir_lote([
    {"id_venda": "V5", "id_funcionario": "F007", "valor": 250.0,
     "timestamp_venda": (INICIO_PLANTAO + timedelta(minutes=80)).isoformat()},
], pasta_sessao)

time.sleep(5)
mostrar_sessoes("Depois de V5 — sessão B continua, sessão A não muda mais")

query_sessao_atual.stop()
query_sessao_fechada.stop()

📌 **Sessão B agora tem 2 vendas**, com fim provisório recalculado para `10:50:00` (última venda 10:20 + 30min). Repare que `sessoes_fechadas` **continua mostrando só a sessão A** — a sessão B nunca chega a fechar nesta simulação, porque nenhum evento seguinte empurra o watermark além do seu fim.

⚠️ **Isso não é um bug, é uma consequência real do modelo:** num pipeline de verdade, a **última** sessão do dia de cada vendedor só é confirmada como fechada quando chega um evento **posterior** (dele, ou de qualquer chave que compartilhe o mesmo relógio de watermark) que empurre o tempo para frente. É por isso que pipelines de sessionização em produção frequentemente injetam eventos de "heartbeat" periódicos, ou aceitam que a sessão do fim do expediente só será gravada quando o primeiro evento do dia seguinte chegar.

## Watermarks: provando a fórmula na prática

A teoria formaliza o watermark como $W(t) = \max(\text{event\_time até } t) - \Delta$: qualquer janela cujo fim seja **menor ou igual** ao watermark atual é considerada fechada e emitida; um evento com `event_time` abaixo do watermark corrente chega tarde demais e é **descartado**.

#### 💡 **Exemplo 5:** Reproduzindo o traçado do watermark, evento a evento

Vamos recriar o mesmo cenário do módulo teórico — tumbling de **5 minutos**, watermark de **10 minutos** — só que aqui os eventos chegam de verdade, **fora de ordem**, e vamos consultar o watermark do Spark após cada um.

🧠 **O pulo do gato:** vamos enviar os eventos numa ordem de **chegada** diferente da ordem de **event time** — é exatamente essa diferença que o watermark existe para tratar. O quarto evento chega com um `event_time` **anterior** a todos os outros (um evento tardio de propósito).

In [ ]:
pasta_watermark = nova_pasta_landing("landing_watermark")

vendas_watermark_stream = (
    spark.readStream.format("json").schema(schema_vendas_stream).load(pasta_watermark)
)

trace_watermark = (
    vendas_watermark_stream
    .withWatermark("timestamp_venda", "10 minutes")
    .groupBy(window(col("timestamp_venda"), "5 minutes"))
    .agg(spark_sum("valor").alias("total_vendido"), count("*").alias("qtd_vendas"))
)

query_watermark = (
    trace_watermark.writeStream
    .format("memory").queryName("trace_watermark")
    .outputMode("append")   # só emite janelas que o watermark já confirmou como fechadas
    .trigger(processingTime="2 seconds").start()
)


def emitir_e_tracar(id_venda: str, minutos: int, segundos: int, valor: float, rotulo: str) -> None:
    emitir_lote([{
        "id_venda": id_venda, "id_funcionario": "F1", "valor": valor,
        "timestamp_venda": (BASE_TIME + timedelta(minutes=minutos, seconds=segundos)).isoformat(),
    }], pasta_watermark)
    time.sleep(5)
    print(f"=== {rotulo} ===")
    spark.sql(
        "SELECT window, total_vendido, qtd_vendas FROM trace_watermark ORDER BY window"
    ).show(truncate=False)

In [ ]:
emitir_e_tracar("A1", 3, 0, 100.0, "Chegada 1: event_time 14:03:00 → watermark = 14:03:00 - 10min = 13:53:00")

In [ ]:
emitir_e_tracar("A2", 12, 0, 200.0, "Chegada 2: event_time 14:12:00 → watermark = 14:12:00 - 10min = 14:02:00 (janela [14:00,14:05) ainda não fecha: 14:05 > 14:02)")

In [ ]:
emitir_e_tracar("A3", 22, 0, 300.0, "Chegada 3: event_time 14:22:00 → watermark = 14:12:00 → janela [14:00,14:05) FECHA e é emitida")

In [ ]:
emitir_e_tracar("A4-tardio", 4, 30, 999.0, "Chegada 4: event_time 14:04:30 (TARDIO!) → pertenceria a [14:00,14:05), mas essa janela já fechou → deve ser DESCARTADO")

📌 **A prova está no `total_vendido`:** antes e depois da chegada 4, a janela `[14:00,14:05)` continua com `total_vendido = 100.0` e `qtd_vendas = 1`. Se o evento tardio (`valor = 999.0`) tivesse sido aceito, o total teria saltado para `1099.0`. Ele não aparece em lugar nenhum — nem somado à janela antiga, nem como uma janela nova — porque `999.0` chegou com um `event_time` já abaixo do watermark corrente (`14:04:30 < 14:12:00`), e o Spark o descarta silenciosamente, exatamente como a regra formal prevê.

✅ **Watermark nunca recua:** mesmo a chegada 4 tendo o `event_time` mais antigo de todos os quatro eventos, o watermark permanece em `14:12:00` — ele é sempre o **máximo histórico** de `event_time` já observado, menos $\Delta$, nunca o valor do evento mais recente isoladamente.

In [ ]:
query_watermark.stop()

## Modos de saída: o que cada exemplo deste notebook realmente usou

| Modo | O que envia ao sink | Onde usamos neste notebook |
|---|---|---|
| `complete` | Reescreve a **tabela de resultado inteira** a cada ciclo | Exemplos 1, 2 e 3 (poucas janelas, e a visão "atual" das sessões no Exemplo 4) |
| `update` | Só as **linhas que mudaram** desde o último ciclo | Não usado — não é suportado por `session_window` (ver observação abaixo) |
| `append` | Só linhas que **nunca mudarão mais** — exige watermark | Sessões fechadas do Exemplo 4, e o traçado completo do Exemplo 5 |

📌 `update` é o modo mais comum para acompanhar janelas `window()` (tumbling/sliding) ainda abertas — mas `session_window` **não suporta `update`**, só `complete`/`append`, como já vimos no Exemplo 4.

🧠 **Regra prática:** poucas chaves/janelas e quer ver tudo sempre → `complete`. Painel ao vivo com `window()` comum e muitas chaves → `update`. Resultado final, gravado uma única vez (Parquet, Kafka de saída) → `append` (com watermark).

## Do file source para Kafka: o que muda (e o que não muda)

Este laboratório não tem um broker Kafka disponível — o `docker-compose.yml` do projeto sobe Spark Standalone + Spark Connect, RustFS (S3) e HDFS, mas nenhum serviço de mensageria. Ainda assim, vale registrar o quão pequena seria a mudança para apontar qualquer exemplo deste notebook para um tópico Kafka de verdade — **ilustrativo, não executado neste ambiente**:

```python
# LEITURA — troca format("json").load(pasta) por format("kafka") + decodificação do payload
eventos_kafka = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka-broker1:9092")
    .option("subscribe", "vendas")
    .option("startingOffsets", "latest")
    .load()
    .selectExpr("CAST(value AS STRING) AS json_str")
    .select(from_json(col("json_str"), schema_vendas_stream).alias("dados"))
    .select("dados.*")
)

# A PARTIR DAQUI, NADA MUDA: os mesmos withWatermark/groupBy/window/session_window
# que já escrevemos funcionam sem alteração nenhuma sobre "eventos_kafka"

# ESCRITA — troca format("memory") por format("kafka"), empacotando as colunas em "value"
query = (
    total_por_janela
    .select(to_json(struct([c for c in total_por_janela.columns])).alias("value"))
    .writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka-broker1:9092")
    .option("topic", "vendas-por-janela")
    .option("checkpointLocation", "/checkpoints/vendas-por-janela/")
    .outputMode("update")
    .start()
)
```

📌 Essa é a ideia central do módulo teórico, agora com prova concreta: **toda a lógica de janelas, watermarks e modos de saída que praticamos aqui roda inalterada em produção** — só a fonte (`readStream`) e o destino (`writeStream`) mudam de `json`/`memory` para `kafka`.

In [ ]:
# Encerra a SparkSession — libera threads e memória
spark.stop()

---
🎉 **Structured Streaming concluído!** Você aprendeu:

- O modelo de **tabela ilimitada**: `readStream`/`writeStream` reaproveitam a mesma API de DataFrame do batch, só a execução muda
- Simular um stream sem Kafka com o **file source** — uma pasta "landing" recebendo arquivos incrementalmente
- **Tumbling windows** (`window(col, "5 minutes")`) para períodos fixos e não sobrepostos, inclusive combinadas com uma chave de negócio (`groupBy(window, id_funcionario)`)
- **Sliding windows** (`window(col, "10 minutes", "5 minutes")`) e por que a janela de sobreposição soma exatamente as janelas tumbling adjacentes
- **Session windows** (`session_window`) para sessões de duração variável por chave — e que elas **exigem** `withWatermark`, com suporte só a `complete`/`append` (não `update`) nesta versão do Spark
- **Watermarks**: reproduzimos, evento a evento, a fórmula $W(t) = \max(\text{event\_time}) - \Delta$ — inclusive provando que um evento tardio é descartado sem alterar o resultado já fechado
- Os três **modos de saída** (`complete`, `update`, `append`) e qual combina com cada tipo de agregação
- Como o mesmo código funcionaria com `format("kafka")` no lugar do file source, sem tocar em nenhuma lógica de janela

📌 Todo o estado desta simulação vive em `../data/streaming/nb12` (git-ignorado) e é limpo automaticamente toda vez que este notebook roda do início.
